### Characterization of metagenomes
Notebook dedicated to characterize the metagenomes pool.

In [1]:
import requests
import json 

def MGnify_search(max_pages):
    """
    Faz a pesquisa de vários IDs de amostras do MGnify. A pesquisa pode ser feita pelo catálogo de genomas do MGnify ou por bioma.
    
    Parâmetros:
    max_pages: número máximo de páginas a buscar.
    """
    print("Início do download dos ids")
    lista_ids = []
    lista_json = []

    for page in range(1, max_pages + 1):
        url = f"https://www.ebi.ac.uk/metagenomics/api/v1/biomes/root:Environmental:Aquatic:Marine/genomes?page={page}"
        resposta = requests.get(url)

        if resposta.status_code != 200:
            print(f"Erro {resposta.status_code} na página {page}")
            break

        dados_json = resposta.json()
        ids_atuais = [item["id"] for item in dados_json.get("data", [])]
        lista_json.extend(dados_json.get("data", []))
        lista_ids.extend(ids_atuais)

        print(f"Página {page}: {len(ids_atuais)} IDs encontrados")
      

    resultado_final_json = {"data": lista_json}
    
    with open("aquatic_download.json", "w") as i:
        json.dump(resultado_final_json, i, indent=4)

    with open("aquatic_ids.txt", "w") as f:
        for id_study in lista_ids:
            f.write(id_study + "\n")

    print(f"\nTotal de {len(lista_ids)} IDs coletados em {page} páginas.")
    print("fim do download dos ids")
    return resultado_final_json, lista_ids

resultado_final_json, lista_ids = MGnify_search(2)

Início do download dos ids
Página 1: 25 IDs encontrados
Página 2: 25 IDs encontrados

Total de 50 IDs coletados em 2 páginas.
fim do download dos ids


In [ ]:
import requests

def genome_list_url(arquivo_saida="links_download_metagenome_characterization.txt"):
    '''
    Faz a pesquisa das urls associadas com cada uma das amostras. Junta tudo em um txt único.

    Parâmetros:
    lista_ids = Lista com o nome dos ids que terão as urls pesquisadas
    arquivo_saida = Nome do arquivo de saída .txt
    '''
    with open("/home/pedro/antismash/aquatic_ids.txt", "r") as f:
        lista_ids = [linha.strip() for linha in f if linha.strip()]
    
    lista_download = []
    for ids in lista_ids:
        print(f"Baixando links para o ID {ids}")
        url = f"https://www.ebi.ac.uk/metagenomics/api/v1/genomes/{ids}/downloads"
        resposta =requests.get(url)
        print(resposta.status_code)
        print(resposta.text[:1000])
        if resposta.status_code == 200:
            url_ids = [item["links"]["self"] for item in resposta.json().get("data", [])]
            lista_download.extend(url_ids)
        else:
            print("Erro na requisição.")
    with open(arquivo_saida, "w") as f:
        for link in lista_download:
            f.write(link + "\n")
    print (lista_download)
    return lista_download

genome_list = genome_list_url()

In [3]:
import requests

def download_genome_url(input, filtro=".fna"):
    """
    Lê um arquivo .txt OU uma lista de links e baixa apenas os arquivos que você quer filtrando pelo tipo de arquivo.
    
    Parâmetros:
    input: caminho do arquivo .txt contendo os links OU uma lista de links.
    filtro: extensão do arquivo a ser baixado. Default: ".fna"
    """
    if isinstance(input, str):
        with open(input, "r") as f:
            links = [linha.strip() for linha in f if linha.strip()]
    elif isinstance(input, list):
        links = [linha.strip() for linha in input if linha.strip()]
    else:
        raise TypeError("O input deve ser um caminho de arquivo (.txt) ou uma lista de links.")

    fna_links = [link for link in links if link.endswith(filtro)] 

    if not fna_links:
        print(f"Nenhum link {filtro} encontrado.") 
        return

    for link in fna_links:
        nome_arquivo = link.split("/")[-1]
        print(nome_arquivo)
        print(f"Baixando {nome_arquivo} ...")

        try:
            resposta = requests.get(link)
            resposta.raise_for_status()
            with open(f"/home/pedro/antismash/genomes_2/{nome_arquivo}", "wb") as f:
                f.write(resposta.content)
            print(f"Download concluído: {nome_arquivo}")
        except requests.RequestException as e:
            print(f"Erro ao baixar {nome_arquivo}: {e}")


download_genomes = download_genome_url("/home/pedro/antismash/scripts_python/links_download.txt")


MGYG000296006.fna
Baixando MGYG000296006.fna ...
Download concluído: MGYG000296006.fna
MGYG000296008.fna
Baixando MGYG000296008.fna ...
Download concluído: MGYG000296008.fna
pan-genome.fna
Baixando pan-genome.fna ...
Download concluído: pan-genome.fna
MGYG000296009.fna
Baixando MGYG000296009.fna ...
Download concluído: MGYG000296009.fna
pan-genome.fna
Baixando pan-genome.fna ...
Download concluído: pan-genome.fna
MGYG000296014.fna
Baixando MGYG000296014.fna ...
Download concluído: MGYG000296014.fna
MGYG000296015.fna
Baixando MGYG000296015.fna ...
Download concluído: MGYG000296015.fna
pan-genome.fna
Baixando pan-genome.fna ...
Download concluído: pan-genome.fna
MGYG000296016.fna
Baixando MGYG000296016.fna ...
Download concluído: MGYG000296016.fna
MGYG000296017.fna
Baixando MGYG000296017.fna ...
Download concluído: MGYG000296017.fna
MGYG000296018.fna
Baixando MGYG000296018.fna ...
Download concluído: MGYG000296018.fna
pan-genome.fna
Baixando pan-genome.fna ...
Download concluído: pan-gen

In [6]:
import pandas as pd
import json

def genome_metadata():
    """
    Carrega o JSON do arquivo e processa os genomas.
    
    Parâmetros:
    caminho_arquivo (str): Caminho para o arquivo JSON
    lista_ids (list): Lista de IDs para filtrar
    """
        
    with open("/home/pedro/antismash/notebooks/aquatic_download.json", "r") as f:
        dados_json = json.load(f)

    
    with open("/home/pedro/antismash/notebooks/aquatic_ids.txt", "r") as f:
        lista_ids = [linha.strip() for linha in f if linha.strip()]
    dados_processados = []
    
    for amostra in dados_json['data']:
        if amostra['id'] in lista_ids:
            linha = {'id': amostra['id']}
            if 'attributes' in amostra:
                linha.update(amostra['attributes'])
            dados_processados.append(linha)
    print(pd.DataFrame(dados_processados))
    df = pd.DataFrame(dados_processados)
    df.to_csv("tabela_ids_MGnify.csv", index=False)
    return pd.DataFrame(dados_processados)

df_metadata = genome_metadata()

               id  genome-id         geographic-range geographic-origin  \
0   MGYG000296006      23178                       []     North America   
1   MGYG000296008      21804          [North America]      not provided   
2   MGYG000296009      20972                       []      not provided   
3   MGYG000296014      31341                       []      not provided   
4   MGYG000296015      32699          [North America]     North America   
5   MGYG000296016      26950                       []     North America   
6   MGYG000296017      32398                       []     North America   
7   MGYG000296018      31081                   [Asia]      not provided   
8   MGYG000296020      33575                       []     North America   
9   MGYG000296023      28696                       []            Europe   
10  MGYG000296030      30338                       []     North America   
11  MGYG000296033      25683                       []            Europe   
12  MGYG000296039      23